# LA2A TVC-LSTM Baseline — Multi-Setting Direct Output

**Google Colab**: Runtime → **GPU**. Open via *File → Open notebook → GitHub*
(`5aola/Virtual-Analogue-Compressor-Modelling`); cell 1 clones the repo and mounts
Drive for the dataset. **Push local changes before running.**

## What this is

The **[`02b_sota_training/train_lstm_diffssl_tvc.ipynb`](../02b_sota_training/train_lstm_diffssl_tvc.ipynb)**
SOTA baseline — the conditioned LSTM from the diffssl paper recipe
(`LSTM32TVC` — `cond_type="tvcond"`, `TVFiLMCond` + sample-rate LSTM,
`0.5·L1 + 0.5·MR-STFT`, AdamW + ReduceLROnPlateau, TBPTT `step_num_samples=4410`)
— **retargeted from Diff-SSL-G-Comp to the Teletronix LA-2A** (SignalTrain 1.1
dataset). The model (published [nablafx](https://github.com/mcomunita/nablafx)
`build_diffssl_tvc_lstm`) and the training system (`DiffSSLTVCLSTMSystem`) are the
*same* `02b_sota_training` code, unchanged — direct wet-audio prediction, not GR.

Only the **dataset, its split, and the conditioning width** differ (the two
intentional LA2A deviations):

```
dry ─ LSTM(tvcond, h=32) ← TVFiLMCond(2 LA2A knobs) ─ Linear ─ wet [direct output]
```

## Dataset — SignalTrain LA2A (`data/LA2A/all/`)

84 **long recordings** (4-20 min each, 44.1 kHz mono float), one per
`(Comp/Limit, Peak Reduction)` setting: `input_<id>_.wav` (dry) +
`target_<id>_LA2A_<Nc>__<cl>__<pr>.wav` (wet). **2 knobs** (`la2a_info.ini`):
Comp/Limit ∈ {0,1} switch, Peak Reduction ∈ {0,5,..,100}. 42 unique settings.
Only the two WAVs are needed — no GR curve (the SOTA baseline is supervised
directly against the wet audio). Crop/batch recipe reuses
[`08_la2a/dataset_la2a.py`](dataset_la2a.py) via
[`08_la2a/dataset_la2a_direct.py`](dataset_la2a_direct.py), which just drops the
GR channel to yield the `(dry, wet, params)` contract the diffssl system expects.

## Split — percentage-based, temporal within each recording (`08_la2a/splits_la2a.py`)

Diff-SSL's song-level / `test_ground_truth` policy does **not** transfer: every
LA2A setting lives in only one (a few in two-three) recording(s), so holding out
whole recordings would delete a setting from training and break the knob
conditioning. Instead we split **temporally by time fraction within each
recording** — the standard LA2A methodology (SignalTrain, Steinmetz TCN,
Comparative-Study, Optical-DRC all test on held-out *audio regions* at the same
settings):

```
per recording:  [0, 0.8) → train    [0.8, 0.9) → val    [0.9, 1.0) → test
```

- **Every setting is in all three splits** → conditioning fully learnable; the
  test set probes generalisation to *unseen audio at known settings* (the LA-2A
  question).
- **No content leakage** — boundaries are by time fraction, identical for every
  recording. Within each region crops are evenly-spaced and capped at
  `crops_per_pair` (same budget as the LA2A GR-predictor run — the two LA2A
  experiments then train/eval on the *same audio windows*).

## Training recipe — diffssl `LSTM32TVC` / `BlackBoxSystemWithTBPTT`

- 3 s crops (`sample_length=132300`), train shuffle + `drop_last`
- LSTM state **reset every batch**; TBPTT sub-steps of `4410` samples inside each crop
- `0.5·L1 + 0.5·MR-STFT`, AdamW + ReduceLROnPlateau
- **Training budget**: fixed **100 epochs**.

### L4-speed tuning (deviates from the diffssl `batch_size=16` recipe)

This LSTM is tiny (~8k params); its cost is the sample-rate recurrence, whose
per-step GPU latency is ~flat across batch sizes, so an L4 is badly
underutilised at batch 16 and epoch wall-clock scales roughly `1/batch_size`.
We therefore run **`batch_size=64`** (≈4× fewer, bigger batches → far better L4
utilisation) with the LR **sqrt-scaled to `2e-3`** (`1e-3·√(64/16)`) to keep the
per-epoch update count healthy at the fixed 100-epoch budget. The data pipeline
also drops the GR-curve compute the parent LA2A dataset does (unused here — see
[`dataset_la2a_direct.py`](dataset_la2a_direct.py)). Set `BATCH_SIZE = 16`,
`LR = 1e-3` below to fall back to the exact diffssl recipe.


In [1]:
# ── 0. Dependencies ──────────────────────────────────────────────────
!pip install -q "numpy>=2.0,<2.6"
!pip install -q torchmetrics soundfile auraloss einops lightning-utilities packaging
!pip install -q --no-deps lightning nablafx

import sys, types

rational = types.ModuleType("rational")
rational.torch = types.ModuleType("rational.torch")
rational.torch.Rational = type("Rational", (), {})
sys.modules["rational"], sys.modules["rational.torch"] = rational, rational.torch

# diffssl nablafx.system imports FAD — stub so we never pull tensorflow/encodec.
fad = types.ModuleType("frechet_audio_distance")
fad.FrechetAudioDistance = type("FrechetAudioDistance", (), {})
sys.modules["frechet_audio_distance"] = fad

import numpy as np, torch
assert np.__version__.startswith("2."), f"numpy {np.__version__} — restart runtime, re-run cell 0"
print(f"numpy {np.__version__}, torch {torch.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.8/249.8 kB 19.1 MB/s eta 0:00:00
numpy 2.0.2, torch 2.11.0+cu128


In [2]:
# ── 1. Mount Drive (dataset) + clone repo from GitHub (code) ─────────
# Model code comes from pip ``nablafx`` (cell 0). This repo supplies the LA2A
# dataset/split (08_la2a) + the diffssl model/system wrappers (02b_sota_training).
# The repo is NOT synced to Drive (only data/ is) — push local changes before
# (re)running; re-running pulls updates.

import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

DRIVE_DATA_ROOT = "/content/drive/Othercomputers/MacBook Air/data/LA2A"
REPO_URL = "https://github.com/5aola/Virtual-Analogue-Compressor-Modelling.git"
REPO_ROOT = "/content/Virtual-Analogue-Compressor-Modelling"

if os.path.isdir(REPO_ROOT):
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" reset --hard origin/main
else:
    !git clone --depth 1 "{REPO_URL}" "{REPO_ROOT}"

DATA_ROOT = DRIVE_DATA_ROOT

# Module dirs: LA2A dataset/split (08_la2a) + the diffssl model/system wrappers
# it reuses unchanged (02b_sota_training). Both go on sys.path, plus the repo
# root for `src`.
LA2A_DIR = os.path.join(REPO_ROOT, "08_la2a")
SOTA_DIR = os.path.join(REPO_ROOT, "02b_sota_training")
assert os.path.isfile(os.path.join(LA2A_DIR, "dataset_la2a_direct.py")), (
    f"Clone failed or stale: {LA2A_DIR}. Did you push local changes?"
)
assert os.path.isfile(os.path.join(SOTA_DIR, "model.py")), (
    f"Missing 02b_sota_training model/system modules: {SOTA_DIR}"
)

OUTPUT_DIR = os.path.join(os.path.dirname(DATA_ROOT), "la2a_tvc_runs")

assert os.path.isdir(os.path.join(DATA_ROOT, "all")), (
    f"No all/ under {DATA_ROOT} — sync the SignalTrain LA2A WAVs to Drive first."
)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Drop cached local modules so a prior run cannot keep stale classes.
for _name in list(sys.modules):
    if _name in ("dataset_la2a", "dataset_la2a_direct", "splits_la2a",
                 "model", "system"):
        del sys.modules[_name]

for p in (REPO_ROOT, SOTA_DIR, LA2A_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"LA2A_DIR   : {LA2A_DIR}")
print(f"SOTA_DIR   : {SOTA_DIR}")
print(f"DATA_ROOT  : {DATA_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 10 (delta 4), reused 9 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 1.08 MiB | 11.55 MiB/s, done.
From https://github.com/5aola/Virtual-Analogue-Compressor-Modelling
   c1bf37d..4ea5695  main       -> origin/main
HEAD is now at 4ea5695 update
REPO_ROOT  : /content/Virtual-Analogue-Compressor-Modelling
LA2A_DIR   : /content/Virtual-Analogue-Compressor-Modelling/08_la2a
SOTA_DIR   : /content/Virtual-Analogue-Compressor-Modelling/02b_sota_training
DATA_ROOT  : /content/drive/Othercomputers/MacBook Air/data/LA2A
OUTPUT_DIR : /content/drive/Othercomputers/MacBook Air/data/la2a_tvc_runs


In [3]:
# ── 2. Cache dataset to Colab local SSD ──────────────────────────────
# Only the input/target WAVs are cached (~29 GB float32). No GR curves needed:
# the SOTA baseline predicts wet audio directly. One-time copy per session.

import shutil
from dataset_la2a_direct import discover_la2a_pairs

LOCAL_DATA_ROOT = "/content/LA2A"
pairs = discover_la2a_pairs(DATA_ROOT)
print(f"Caching {len(pairs)} recordings (input+target WAVs) → {LOCAL_DATA_ROOT}")

def _mirror(src, dst):
    src, dst = Path(src), Path(dst)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

for i, p in enumerate(pairs, 1):
    for key in ("dry", "wet"):
        _mirror(p[key], Path(LOCAL_DATA_ROOT) / Path(p[key]).relative_to(DATA_ROOT))
    if i % 10 == 0 or i == len(pairs):
        print(f"  cached {i}/{len(pairs)}")

DATA_ROOT = LOCAL_DATA_ROOT
print(f"Using local cache: {DATA_ROOT}")

Caching 84 recordings (input+target WAVs) → /content/LA2A
  cached 10/84
  cached 20/84
  cached 30/84
  cached 40/84
  cached 50/84
  cached 60/84
  cached 70/84
  cached 80/84
  cached 84/84
Using local cache: /content/LA2A


In [4]:
# ── 3. Imports & hyper-parameters (LSTM32TVC on LA2A) ───────────────

import importlib
import json
from datetime import datetime

import lightning as pl
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint, TQDMProgressBar
from lightning.pytorch.loggers import CSVLogger, TensorBoardLogger

import dataset_la2a as _dataset_la2a
importlib.reload(_dataset_la2a)
import dataset_la2a_direct as _dataset_la2a_direct
importlib.reload(_dataset_la2a_direct)
from dataset_la2a_direct import (
    RMS_WINDOW, SAMPLE_LENGTH, SAMPLE_RATE,
    La2aDirectCropDataModule, discover_la2a_pairs,
)  # BATCH_SIZE is set explicitly below (L4 tuning), not imported

import splits_la2a as _splits_la2a
importlib.reload(_splits_la2a)
from splits_la2a import (
    LA2A_PARAM_ORDER, LA2A_PARAM_RANGES, build_la2a_split_manifest,
)

import model as _model
importlib.reload(_model)
from model import build_diffssl_tvc_lstm

import system as _system
importlib.reload(_system)
from system import DiffSSLTVCLSTMSystem

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "WARNING: CPU runtime")

# -- split: percentage-based, temporal within each recording (LA2A convention) --
SPLIT_SEED = 42
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.8, 0.1, 0.1
# evenly-spaced crops per recording per split — same budget as the LA2A GR run,
# so both LA2A experiments train/eval on the exact same audio windows.
CROPS_PER_PAIR = {"train": 24, "val": 4, "test": 8}

# -- training recipe: diffssl LSTM32TVC / BlackBoxSystemWithTBPTT --
# L4-speed tuning: the diffssl recipe is batch 16 / LR 1e-3. This LSTM is tiny
# and the L4 is idle most of the recurrence, so we run a bigger batch (epoch
# wall-clock ~ 1/batch) with the LR sqrt-scaled to keep 100-epoch convergence.
# Set BATCH_SIZE=16, LR=1e-3 to reproduce the exact diffssl recipe.
BATCH_SIZE = 64                  # diffssl uses 16; 64 far better utilises the L4
LR = 2e-3                        # sqrt-scaled: 1e-3 * sqrt(64/16)
MAX_EPOCHS = 100
STEP_NUM_SAMPLES = 4410          # diffssl LSTM TBPTT sub-step (0.1 s @ 44.1 kHz)

# -- model (diffssl LSTM32TVC; 2 LA2A knobs instead of 4 diffssl params) --
HIDDEN_SIZE = 32                 # LSTM32TVC; set 96 for LSTM96TVC
NUM_LAYERS = 1
COND_TYPE = "tvcond"
COND_BLOCK_SIZE = 128
COND_NUM_LAYERS = 1
NUM_CONTROLS = 2                 # LA2A: [comp_limit, peak_reduction]

print(f"Crop {SAMPLE_LENGTH} ({SAMPLE_LENGTH/SAMPLE_RATE:.2f}s) | batch {BATCH_SIZE} | "
      f"TBPTT step {STEP_NUM_SAMPLES} ({STEP_NUM_SAMPLES/SAMPLE_RATE:.2f}s)")

RUN_TAG = "la2a_lstm32_tvc_multisetting"
RESUME_RUN = None

NVIDIA L4
Crop 132300 (3.00s) | batch 64 | TBPTT step 4410 (0.10s)


In [5]:
# ── 4. Preview split — percentage-based, temporal within each recording ─
# Every (comp_limit, peak_reduction) setting appears in train/val/test on
# DISJOINT temporal regions of its recording: conditioning is fully learnable
# and the test set is unseen audio at known settings (the LA2A convention).

pairs = discover_la2a_pairs(DATA_ROOT)
preview = build_la2a_split_manifest(
    pairs, seed=SPLIT_SEED, sample_length=SAMPLE_LENGTH,
    train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, test_frac=TEST_FRAC,
    crops_per_pair=CROPS_PER_PAIR,
)

print(f"Recordings : {len(preview.pairs)}")
print(f"Settings   : {len(preview.settings)} unique [comp_limit, peak_reduction]")
print(f"  comp/limit values : {sorted({s[0] for s in preview.settings})}")
print(f"  peak_reduction    : {sorted({s[1] for s in preview.settings})}")
print(f"Regions    : train[0,{TRAIN_FRAC}) val[{TRAIN_FRAC},{round(TRAIN_FRAC+VAL_FRAC,3)}) "
      f"test[{round(TRAIN_FRAC+VAL_FRAC,3)},1) of every recording")
print(f"Crop totals: {preview.crop_counts}  (<= {CROPS_PER_PAIR} per recording)")
mins = {k: v * SAMPLE_LENGTH / SAMPLE_RATE / 60 for k, v in preview.crop_counts.items()}
print("Audio (min): " + "  ".join(f"{k}={mins[k]:.1f}" for k in ("train", "val", "test")))

assert all(preview.crop_counts[k] > 0 for k in ("train", "val", "test")), "empty split!"

Recordings : 84
Settings   : 42 unique [comp_limit, peak_reduction]
  comp/limit values : [0, 1]
  peak_reduction    : [0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]
Regions    : train[0,0.8) val[0.8,0.9) test[0.9,1) of every recording
Crop totals: {'train': 2016, 'val': 336, 'test': 672}  (<= {'train': 24, 'val': 4, 'test': 8} per recording)
Audio (min): train=100.8  val=16.8  test=33.6


In [6]:
# ── 5. Build diffssl model (LSTM32TVC / LSTM96TVC) on 2 LA2A knobs ──

model = build_diffssl_tvc_lstm(
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_controls=NUM_CONTROLS,
    cond_block_size=COND_BLOCK_SIZE,
    cond_num_layers=COND_NUM_LAYERS,
)
n_params = sum(p.numel() for p in model.parameters())
print(f"BlackBoxModel + LSTM(tvcond, h={HIDDEN_SIZE}): {n_params:,} params "
      f"({NUM_CONTROLS} LA2A knobs; diffssl 4-knob run was 8,033)")
print(model.processor)


BlackBoxModel:
LSTM(
  (cond_nn): TVFiLMCond(
    (pool): MaxPool1d(kernel_size=128, stride=128, padding=0, dilation=1, ceil_mode=False)
    (lstm): LSTM(3, 16)
  )
  (lstm): LSTM(17, 32)
  (lin): Linear(in_features=32, out_features=1, bias=True)
)

BlackBoxModel + LSTM(tvcond, h=32): 7,905 params (2 LA2A knobs; diffssl 4-knob run was 8,033)
LSTM(
  (cond_nn): TVFiLMCond(
    (pool): MaxPool1d(kernel_size=128, stride=128, padding=0, dilation=1, ceil_mode=False)
    (lstm): LSTM(3, 16)
  )
  (lstm): LSTM(17, 32)
  (lin): Linear(in_features=32, out_features=1, bias=True)
)


In [7]:
# ── 6. Train ─────────────────────────────────────────────────────────

torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

assert DATA_ROOT.startswith("/content/"), "Run the cache cell first (cell 2)."

# Multi-worker data loading: the model is tiny (8k params), so without this the
# GPU starves on the per-item soundfile seeks (dry + wet). Cap at 8.
NUM_WORKERS = min(8, os.cpu_count() or 2)
print(f"DataLoader num_workers: {NUM_WORKERS}")

if RESUME_RUN:
    RUN_NAME = RESUME_RUN
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = os.path.join(RUN_DIR, "checkpoints", "last.ckpt")
    print(f"RESUMING: {RUN_NAME}")
else:
    RUN_NAME = f"la2a_tvc_{datetime.now():%Y%m%d_%H%M%S}_{RUN_TAG}"
    RUN_DIR = os.path.join(OUTPUT_DIR, RUN_NAME)
    _resume_ckpt = None
    print(f"NEW run: {RUN_NAME}")

os.makedirs(RUN_DIR, exist_ok=True)
split_path = os.path.join(RUN_DIR, "split_manifest.json")

dm = La2aDirectCropDataModule(
    data_root=DATA_ROOT, sample_length=SAMPLE_LENGTH, sample_rate=SAMPLE_RATE,
    batch_size=BATCH_SIZE, split_seed=SPLIT_SEED,
    train_frac=TRAIN_FRAC, val_frac=VAL_FRAC, test_frac=TEST_FRAC,
    crops_per_pair=CROPS_PER_PAIR, rms_window=RMS_WINDOW,
    split_manifest_path=split_path, num_workers=NUM_WORKERS,
)
dm.setup()
print(f"Train/val/test crops: {len(dm.train_dataset)} / {len(dm.val_dataset)} / {len(dm.test_dataset)}")
print(f"Batches/epoch (train): {len(dm.train_dataloader())}  (batch_size={BATCH_SIZE})")
print(f"Split manifest: {split_path}")

with open(os.path.join(RUN_DIR, "hparams.json"), "w") as f:
    json.dump({
        "approach": "diffssl_direct_output_lstm_tvcond",
        "model_type": "nablafx_diffssl_LSTM_tvcond",
        "model_ref": f"experiments/LSTM{HIDDEN_SIZE}TVC/config.yaml (retargeted to LA2A)",
        "dataset": "SignalTrain-LA2A",
        "setting": "multi (all 42 comp_limit x peak_reduction settings)",
        "conditioning": "tvcond (TVFiLMCond)",
        "sample_rate": SAMPLE_RATE,
        "sample_length": SAMPLE_LENGTH,
        "batch_size": BATCH_SIZE,
        "step_num_samples": STEP_NUM_SAMPLES,
        "param_order": LA2A_PARAM_ORDER,
        "param_ranges": LA2A_PARAM_RANGES,
        "split_seed": SPLIT_SEED,
        "split_policy": "temporal_within_recording (every setting in train/val/test; test = unseen audio regions)",
        "split_fracs": {"train": TRAIN_FRAC, "val": VAL_FRAC, "test": TEST_FRAC},
        "crops_per_pair": CROPS_PER_PAIR,
        "num_settings": len(dm.split.settings),
        "num_recordings": len(dm.split.pairs),
        "crop_counts": dm.split.crop_counts,
        "model": {
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "cond_type": COND_TYPE,
            "cond_block_size": COND_BLOCK_SIZE,
            "cond_num_layers": COND_NUM_LAYERS,
            "num_controls": NUM_CONTROLS,
            "num_params": n_params,
        },
        "lr": LR,
        "max_epochs": MAX_EPOCHS,
        "loss": "0.5*L1 + 0.5*MR-STFT",
        "optimizer": "adamw + reducelronplateau(0.5,p20)",
        "training": "la2a_crop_batches + tbptt_substeps (reset each batch)",
        "l4_speed_tuning": (
            f"batch {BATCH_SIZE} + LR {LR} (sqrt-scaled) vs diffssl batch 16 / LR 1e-3; "
            "epoch wall-clock ~ 1/batch for this sample-rate LSTM"
        ),
    }, f, indent=2)

system = DiffSSLTVCLSTMSystem(
    model=model,
    lr=LR,
    step_num_samples=STEP_NUM_SAMPLES,
)

ckpt_dir = os.path.join(RUN_DIR, "checkpoints")
callbacks = [
    ModelCheckpoint(
        dirpath=ckpt_dir, monitor="loss/val", mode="min", save_top_k=3,
        save_last=True, filename="best-{epoch:03d}-{step}",
        auto_insert_metric_name=False,
    ),
    LearningRateMonitor(logging_interval="epoch"),
    TQDMProgressBar(refresh_rate=10),
]
loggers = [
    TensorBoardLogger(save_dir=RUN_DIR, name="tb", version=""),
    CSVLogger(save_dir=RUN_DIR, name="csv", version=""),
]

# NOTE: DiffSSLTVCLSTMSystem uses manual optimization (automatic_optimization
# = False), so Lightning gradient clipping is disabled — the TBPTT loop steps
# the optimizer itself. Do not pass gradient_clip_val here.
trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS, accelerator="gpu", devices=1,
    callbacks=callbacks, logger=loggers, log_every_n_steps=10,
)

trainer.fit(system, dm, ckpt_path=_resume_ckpt)
print(f"Best val loss: {callbacks[0].best_model_score:.6f}")
print(f"Best ckpt    : {callbacks[0].best_model_path}")

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


DataLoader num_workers: 8
NEW run: la2a_tvc_20260708_000134_la2a_lstm32_tvc_multisetting
Recordings     : 84
Settings       : 42 unique (comp_limit, peak_reduction)
Split fractions: train=0.8 val=0.1 test=0.1
Crops per rec  : {'train': 24, 'val': 4, 'test': 8}
Crop totals    : {'train': 2016, 'val': 336, 'test': 672}
La2aCropDataset[train]: 2016 crops from 84 recordings  [<= 24/rec, sample_length=132300, 100.8 min audio]
La2aCropDataset[val]: 336 crops from 84 recordings  [<= 4/rec, sample_length=132300, 16.8 min audio]
La2aCropDataset[test]: 672 crops from 84 recordings  [<= 8/rec, sample_length=132300, 33.6 min audio]
Train/val/test crops: 2016 / 336 / 672
Batches/epoch (train): 31  (batch_size=64)
Split manifest: /content/drive/Othercomputers/MacBook Air/data/la2a_tvc_runs/la2a_tvc_20260708_000134_la2a_lstm32_tvc_multisetting/split_manifest.json


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/core/optimizer.py:317: The lr scheduler dict contains the key(s) ['monitor'], but the keys will be ignored. You need to call `lr_scheduler.step()` manually in manual optimization.


┏━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name   ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model  │ BlackBoxModel           │  7.9 K │ train │     0 │
│ 1 │ l1     │ L1Loss                  │      0 │ train │     0 │
│ 2 │ mrstft │ MultiResolutionSTFTLoss │      0 │ train │     0 │
└───┴────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 7.9 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 7.9 K                                                                                                
Total estimated model params size (MB): 0.032                                                                      
Modules in train mode: 28                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=100` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=100` reached.


Best val loss: 0.251247
Best ckpt    : /content/drive/Othercomputers/MacBook Air/data/la2a_tvc_runs/la2a_tvc_20260708_000134_la2a_lstm32_tvc_multisetting/checkpoints/best-061-57660.ckpt


In [8]:
# ── 7. Test (held-out temporal regions — unseen audio at known settings) ─

best_ckpt = callbacks[0].best_model_path or os.path.join(ckpt_dir, "last.ckpt")
print(f"Testing with: {best_ckpt}")
trainer.test(system, dm, ckpt_path=best_ckpt)

INFO: Restoring states from the checkpoint path at /content/drive/Othercomputers/MacBook Air/data/la2a_tvc_runs/la2a_tvc_20260708_000134_la2a_lstm32_tvc_multisetting/checkpoints/best-061-57660.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Restoring states from the checkpoint path at /content/drive/Othercomputers/MacBook Air/data/la2a_tvc_runs/la2a_tvc_20260708_000134_la2a_lstm32_tvc_multisetting/checkpoints/best-061-57660.ckpt
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: Loaded model weights from the checkpoint at /content/drive/Othercomputers/MacBook Air/data/la2a_tvc_runs/la2a_tvc_20260708_000134_la2a_lstm32_tvc_multisetting/checkpoints/best-061-57660.ckpt
INFO:lightning.pytorch.utilities.rank_zero:Loaded model weights from the checkpoint at /content/drive/Othercomputers/MacBook Air/data/la2a_tvc_runs/la2a_tvc_20260708_000134_la2a_lstm32_tvc_multisetting/checkpoints/best-061-57660.ckpt


Testing with: /content/drive/Othercomputers/MacBook Air/data/la2a_tvc_runs/la2a_tvc_20260708_000134_la2a_lstm32_tvc_multisetting/checkpoints/best-061-57660.ckpt


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         esr/test          │    2.0672671794891357     │
│         loss/test         │    0.2539639174938202     │
│       loss/test_fd        │    0.4942050278186798     │
│       loss/test_td        │   0.013722775503993034    │
│         mae/test          │   0.013722775503993034    │
│         mse/test          │   0.0017034381162375212   │
│         rmse/test         │   0.012392081320285797    │
└───────────────────────────┴───────────────────────────┘

[{'loss/test': 0.2539639174938202,
  'loss/test_td': 0.013722775503993034,
  'loss/test_fd': 0.4942050278186798,
  'mae/test': 0.013722775503993034,
  'mse/test': 0.0017034381162375212,
  'esr/test': 2.0672671794891357,
  'rmse/test': 0.012392081320285797}]

In [9]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb"

<IPython.core.display.Javascript object>